# Software Documentation Assistant: Exploration & Indexing
## Week 3 | Retrieval-Augmented Generation Applications (Part C)

This notebook demonstrates the end-to-end ingestion and indexing pipeline for developer technical documentation:
1. Document loading with YAML frontmatter parsing
2. Code-aware hierarchical chunking preserving Markdown code blocks
3. Dense vector embeddings with SentenceTransformers (`all-MiniLM-L6-v2`)
4. Sparse BM25 lexical indexing
5. Index persistence and verification

In [1]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

Project root: C:\Users\ROSHAN ALEXANDER\Documents\Projects\Software-Documentation-Assistant


### 1. Ingest Technical Documents
We load our curated technical documentation corpus across FastAPI (v0.115 and legacy v0.90) and Pydantic (v2.0 and v1.10).

In [2]:
from src.document_loader import DocumentLoader
from src.config import settings

loader = DocumentLoader(settings.DOCS_DIR)
docs = loader.load_all()
print(f"Loaded {len(docs)} technical documentation files:\n")
for d in docs:
    print(f"- [{d.doc_id}] {d.title} (Framework: {d.framework}, Ver: {d.version}, Chars: {d.char_count})")

Loaded 13 technical documentation files:

- [DOC-FA-009] Async Database Integration with SQLAlchemy 2.0 in FastAPI (Framework: FastAPI, Ver: v0.115, Chars: 2257)
- [DOC-FA-006] Asynchronous Background Tasks in FastAPI (Framework: FastAPI, Ver: v0.115, Chars: 2314)
- [DOC-FA-003] Modern Dependency Injection with typing.Annotated in FastAPI (Framework: FastAPI, Ver: v0.115, Chars: 2669)
- [DOC-FA-008] Custom Error Handling and Exception Handlers in FastAPI (Framework: FastAPI, Ver: v0.115, Chars: 2371)
- [DOC-FA-001] FastAPI Lifespan Events and Application State Management (Framework: FastAPI, Ver: v0.115, Chars: 2902)
- [DOC-FA-005] Middleware Architecture and Custom BaseHTTPMiddleware in FastAPI (Framework: FastAPI, Ver: v0.115, Chars: 2399)
- [DOC-FA-004] OAuth2 Password Flow and JWT Authentication in FastAPI (Framework: FastAPI, Ver: v0.115, Chars: 3718)
- [DOC-FA-007] Streaming Responses, Server-Sent Events (SSE), and WebSockets (Framework: FastAPI, Ver: v0.115, Chars: 2415)
- [DOC-

### 2. Code-Aware Hierarchical Chunking
Notice that code blocks (` ```python ... ``` `) are kept intact to preserve syntactic validity.

In [3]:
from src.chunker import MarkdownCodeAwareChunker

chunker = MarkdownCodeAwareChunker(target_chunk_size=settings.CHUNK_SIZE, chunk_overlap=settings.CHUNK_OVERLAP)
chunks = chunker.chunk_all(docs)
code_chunks = [c for c in chunks if c.has_code]

print(f"Total chunks generated: {len(chunks)}")
print(f"Chunks containing code examples: {len(code_chunks)} ({len(code_chunks)/len(chunks)*100:.1f}%)")

# Inspect a sample code chunk
sample = code_chunks[0]
print(f"\nSample Chunk ID: {sample.chunk_id}")
print(f"Citation: {sample.citation}")
print("Content Preview:")
print(sample.content[:300] + "...")

Total chunks generated: 76
Chunks containing code examples: 10 (13.2%)

Sample Chunk ID: DOC-FA-006_chk03
Citation: [Doc: DOC-FA-006 | Section: 'Code Example: Email and Audit Logging' | Ver: v0.115]
Content Preview:
```python
import asyncio
from typing import Annotated
from fastapi import FastAPI, BackgroundTasks, Depends
from pydantic import BaseModel, EmailStr

app = FastAPI(title="Background Tasks API", version="0.115.0")

class UserSignup(BaseModel):
    username: str
    email: EmailStr

async def send_wel...


### 3. Build Hybrid Vector Store
Compute normalized dense vectors and construct the BM25 lexical inverted index.

In [4]:
from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline()
index_stats = pipeline.build_or_load_index(force_reindex=False)
print("Index status:", index_stats)
print("Pipeline stats:", pipeline.get_stats())

[INFO] Vector store loaded from C:\Users\ROSHAN ALEXANDER\Documents\Projects\Software-Documentation-Assistant\results\index_cache (75 chunks).
[INFO] Loaded existing index (75 chunks) in 0.01s.
Index status: {'status': 'loaded_from_cache', 'total_chunks': 75, 'duration_seconds': 0.006488799999715411}
Pipeline stats: {'total_chunks': 75, 'code_chunks': 10, 'prose_chunks': 65, 'unique_versions': ['v0.90-legacy', 'v0.115', 'v1.10-legacy', 'v2.0'], 'frameworks': ['FastAPI', 'Pydantic'], 'categories': ['Realtime & Streaming', 'Dependencies', 'Error Handling', 'Testing', 'Validation', 'Async Processing', 'Database', 'Data Modeling', 'Middleware', 'Security', 'Lifecycle']}
